# Run 4 - TF-IDF + CamemBERT + SVM


In [ ]:
!pip install -q pandas scikit-learn gensim transformers torch scipy sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 56.9 MB/s eta 0:00:00


In [ ]:
import gc
import re
import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.svm import SVC
from transformers import AutoTokenizer, AutoModel
from google.colab import files

print("Imports OK")
print(f"GPU disponible : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("ATTENTION : pas de GPU ! Va dans Runtime -> Change runtime type -> T4 GPU")

Imports OK
GPU disponible : True


In [ ]:
# Upload train.csv ET test.csv
uploaded = files.upload()

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train : {len(train)} recettes")
print(f"Test  : {len(test)} recettes")

assert len(train) == 12473, f"ERREUR train incomplet : {len(train)} lignes au lieu de 12473"
assert len(test)  == 1388,  f"ERREUR test incomplet : {len(test)} lignes au lieu de 1388"

print("Dataset OK")
print(train["type"].value_counts())

y_train = train["type"]
y_test  = test["type"]

Saving test.csv to test.csv
Saving train.csv to train.csv
Train : 12473 recettes
Test  : 1388 recettes
Dataset OK
type
Plat principal    5802
Dessert           3762
Entrée            2909
Name: count, dtype: int64


In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zàâäéèêëîïôöùûüçœæ\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def build_text(df, columns):
    return df[columns].fillna("").astype(str).agg(" ".join, axis=1)

# Texte complet pour TF-IDF
x_train_text = build_text(train, ["titre", "ingredients", "recette"]).apply(preprocess)
x_test_text  = build_text(test,  ["titre", "ingredients", "recette"]).apply(preprocess)

# Titre seul pour CamemBERT
x_train_titre = train["titre"].fillna("").apply(preprocess)
x_test_titre  = test["titre"].fillna("").apply(preprocess)

print("Preprocessing OK")

Preprocessing OK


In [ ]:
# TF-IDF avec meilleurs params (Grid Search Run 2)
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2)

x_train_tfidf = tfidf.fit_transform(x_train_text)
x_test_tfidf  = tfidf.transform(x_test_text)

print(f"TF-IDF train : {x_train_tfidf.shape}")
print(f"TF-IDF test  : {x_test_tfidf.shape}")

TF-IDF train : (12473, 10000)
TF-IDF test  : (1388, 10000)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

tokenizer  = AutoTokenizer.from_pretrained("camembert-base")
camembert  = AutoModel.from_pretrained("camembert-base").to(device)
camembert.eval()
print("CamemBERT charge")

Device : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CamemBERT charge


In [ ]:
def encode_texts(texts, tokenizer, model, device, max_length=128):
    all_embeddings = []
    texts = list(texts)
    batch_size = 32 if torch.cuda.is_available() else 16

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs)

        emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.append(emb)

        del inputs, out, emb
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size, len(texts))}/{len(texts)}")

    return np.vstack(all_embeddings).astype(np.float32)

print("Extraction CamemBERT train...")
x_train_bert = encode_texts(x_train_titre, tokenizer, camembert, device)
print("Extraction CamemBERT test...")
x_test_bert  = encode_texts(x_test_titre,  tokenizer, camembert, device)

print(f"CamemBERT train : {x_train_bert.shape}")
print(f"CamemBERT test  : {x_test_bert.shape}")

Extraction CamemBERT train...
  32/12473
  672/12473
  1312/12473
  1952/12473
  2592/12473
  3232/12473
  3872/12473
  4512/12473
  5152/12473
  5792/12473
  6432/12473
  7072/12473
  7712/12473
  8352/12473
  8992/12473
  9632/12473
  10272/12473
  10912/12473
  11552/12473
  12192/12473
Extraction CamemBERT test...
  32/1388
  672/1388
  1312/1388
CamemBERT train : (12473, 768)
CamemBERT test  : (1388, 768)


In [ ]:
# Combinaison TF-IDF + CamemBERT
x_train_combined = hstack([x_train_tfidf, csr_matrix(x_train_bert)], format="csr")
x_test_combined  = hstack([x_test_tfidf,  csr_matrix(x_test_bert)],  format="csr")

print(f"Combined train : {x_train_combined.shape}")
print(f"Combined test  : {x_test_combined.shape}")

Combined train : (12473, 10768)
Combined test  : (1388, 10768)


In [ ]:
print("Entrainement SVM...")
svm = SVC(kernel="linear", C=1.0, random_state=42)
svm.fit(x_train_combined, y_train)
y_pred = svm.predict(x_test_combined)

f1  = f1_score(y_test, y_pred, average="macro")
acc = accuracy_score(y_test, y_pred)

print("=== Run 4 - TF-IDF + CamemBERT + SVM ===")
print(classification_report(y_test, y_pred))
print(f"Accuracy : {acc:.4f}")
print(f"F1 macro : {f1:.4f}")

Entrainement SVM...
=== Run 4 - TF-IDF + CamemBERT + SVM ===
                precision    recall  f1-score   support

       Dessert       0.99      0.99      0.99       407
        Entrée       0.78      0.73      0.76       337
Plat principal       0.86      0.89      0.88       644

      accuracy                           0.88      1388
     macro avg       0.88      0.87      0.87      1388
  weighted avg       0.88      0.88      0.88      1388

Accuracy : 0.8804
F1 macro : 0.8729


In [ ]:
print("=" * 55)
print("RECAP GENERAL - tous les runs")
print("=" * 55)
recap = [
    ("Run 1  Baseline",              0.211),
    ("Run 2  TF-IDF + SVM",          0.863),
    ("Run 3a Word2Vec + SVM",         0.844),
    ("Run 3b CamemBERT + SVM",        0.708),
    ("Run 4  TF-IDF + CamemBERT + SVM", round(f1, 3)),
]
for name, score in recap:
    bar = chr(9608) * int(score * 30)
    print(f"{name:<40} {score:.3f}  {bar}")

pd.Series(y_pred, name="prediction").to_csv("predictions_run4.csv", index=False)
files.download("predictions_run4.csv")

RECAP GENERAL - tous les runs
Run 1  Baseline                          0.211  ██████
Run 2  TF-IDF + SVM                      0.863  █████████████████████████
Run 3a Word2Vec + SVM                    0.844  █████████████████████████
Run 3b CamemBERT + SVM                   0.708  █████████████████████
Run 4  TF-IDF + CamemBERT + SVM          0.873  ██████████████████████████


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>